In [ ]:
# --- Vishalam setup: find the code dataset among the inputs, unpack, install (no secrets) ---
import glob, os, shutil, subprocess, sys, tarfile, time
T0 = time.time()
def hours_left(limit=11.2):  # Kaggle stops sessions at 12 h; keep a margin for saving outputs
    return max(0.25, limit - (time.time() - T0) / 3600)
WORK, INPUT, CODE = "/kaggle/working", "/kaggle/input", "/tmp/vishalam"

def inputs_with(marker, hint=""):
    """Input directories that contain `marker` (a relative path) and whose path contains `hint`."""
    hits = glob.glob(f"{INPUT}/**/{marker}", recursive=True)
    dirs = [h[: -len(marker)].rstrip("/") for h in hits]
    return sorted(d for d in set(dirs) if hint in d)

if not os.path.isdir(CODE):
    tars = glob.glob(f"{INPUT}/**/vishalam_code.tar.gz", recursive=True)
    if tars:
        os.makedirs(CODE)
        with tarfile.open(tars[0]) as t:
            t.extractall(CODE, filter="data")
    else:  # Kaggle sometimes unpacks archives itself
        roots = [d for d in inputs_with("vishalam/tokenizer/bpe.py") if os.path.exists(f"{d}/requirements.txt")]
        if not roots:
            raise SystemExit("code dataset not attached - run tools/kaggle_driver.py push-code and re-push this kernel")
        shutil.copytree(roots[0], CODE)
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("ready:", open("VERSION").read().strip() if os.path.exists("VERSION") else "code unpacked")


In [ ]:
# --- GPU guard: stop before spending quota if Kaggle gave us the wrong accelerator ---
import torch
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", names)
if not names or not all("T4" in n for n in names):
    raise SystemExit(f"Expected T4 GPUs, got {names}. Set Accelerator -> 'GPU T4 x2' and run again.")


In [ ]:
# --- locate the tokenizer and all token-shard dirs among the attached kernel outputs ---
DATA_DIRS = inputs_with("manifest.json", "vishalam-data-")
TOKENIZER = inputs_with("tokenizer.json", "vishalam-data-1")[0] + "/tokenizer.json"
print("shard dirs:", DATA_DIRS)


In [ ]:
# --- 350M speed test on 2xT4: real data + real training code, a few settings, ~20 steps each ---
import re, subprocess
SETTINGS = [  # micro batch per GPU, activation checkpointing
    (1, False), (2, True), (2, False), (4, True),
]
summary = []
for micro, ckpt in SETTINGS:
    tag = f"micro={micro} ckpt={ckpt}"
    cmd = (f"torchrun --standalone --nproc_per_node={torch.cuda.device_count()} -m vishalam.train.pretrain "
           f"--config configs/vishalam-350m.yaml --data-dirs {','.join(DATA_DIRS)} --tokenizer {TOKENIZER} "
           f"--out /tmp/bench_{micro}_{int(ckpt)} --stop-at-step 20 --set global_batch_seqs=32 "
           f"--set micro_batch={micro} --set grad_checkpointing={str(ckpt).lower()} --set eval_every=100000")
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800)
        out = r.stdout + r.stderr
    except subprocess.TimeoutExpired:
        out = "TIMEOUT"
    steps = [l for l in out.splitlines() if l.startswith("step ")]
    print(f"==== {tag}\n" + ("\n".join(steps) or out[-1500:]), flush=True)
    m = re.search(r"([\d,]+) tok/s mem ([\d.]+)GB", steps[-1]) if steps else None
    summary.append(f"{tag}: {m.group(1)} tok/s, {m.group(2)} GB" if m else f"{tag}: FAILED (out of memory?)")
print("\n==== SUMMARY (350M, 2xT4, Muon) ====\n" + "\n".join(summary))
